### JAX Fragrance Notebook

In [195]:
! pip install -q "transformers[flax]"
! pip install -U flax jax jaxlib
import jax
import jax.numpy as jnp
import random
import pandas as pd
import ast
import threading
import pickle

  Using cached flax-0.8.5-py3-none-any.whl.metadata (10 kB)
  Using cached jax-0.4.30-py3-none-any.whl.metadata (22 kB)
  Using cached jaxlib-0.4.30-cp39-cp39-win_amd64.whl.metadata (1.1 kB)
Using cached flax-0.8.5-py3-none-any.whl (731 kB)
Using cached jax-0.4.30-py3-none-any.whl (2.0 MB)
Using cached jaxlib-0.4.30-cp39-cp39-win_amd64.whl (51.9 MB)
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.4.13
    Uninstalling jaxlib-0.4.13:
      Successfully uninstalled jaxlib-0.4.13
  Attempting uninstall: jax
    Found existing installation: jax 0.4.13
    Uninstalling jax-0.4.13:
      Successfully uninstalled jax-0.4.13
  Attempting uninstall: flax
    Found existing installation: flax 0.7.0
    Uninstalling flax-0.7.0:
      Successfully uninstalled flax-0.7.0


##### Load CSV into Data Frames
- unzip `perfumes_table.csv`

In [ ]:
df = pd.read_csv('./perfumes_table.csv')
url_df = pd.DataFrame({'urls': [u for u in df['url'].values]}) 
designer_df = pd.DataFrame({'designers': [d for d in df['designer'].values]}) 
titles_df = pd.DataFrame({'titles': [title for title in df['title'].values]})
reviews_df = pd.DataFrame({'reviews': [list(set(ast.literal_eval(c))) for c in df['reviews']]})
notes_df = pd.DataFrame({'notes': [str(sorted(ast.literal_eval(notes))) for notes in df['notes'].values]})
print("CSV Columns:", df.columns)

CSV Columns: Index(['rating', 'notes', 'designer', 'reviews', 'description', 'url',
       'title'],
      dtype='object')


- Load a pre-trained embedding model into Flax from HF's `transformers` library

In [197]:
from transformers import AutoTokenizer, FlaxAutoModel

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large-instruct")
model = FlaxAutoModel.from_pretrained("intfloat/multilingual-e5-large-instruct")

Some of the weights of FlaxXLMRobertaModel were initialized in float16 precision from the model checkpoint at intfloat/multilingual-e5-large-instruct:
[('embeddings', 'LayerNorm', 'bias'), ('embeddings', 'LayerNorm', 'scale'), ('embeddings', 'position_embeddings', 'embedding'), ('embeddings', 'token_type_embeddings', 'embedding'), ('embeddings', 'word_embeddings', 'embedding'), ('encoder', 'layer', '0', 'attention', 'output', 'LayerNorm', 'bias'), ('encoder', 'layer', '0', 'attention', 'output', 'LayerNorm', 'scale'), ('encoder', 'layer', '0', 'attention', 'output', 'dense', 'bias'), ('encoder', 'layer', '0', 'attention', 'output', 'dense', 'kernel'), ('encoder', 'layer', '0', 'attention', 'self', 'key', 'bias'), ('encoder', 'layer', '0', 'attention', 'self', 'key', 'kernel'), ('encoder', 'layer', '0', 'attention', 'self', 'query', 'bias'), ('encoder', 'layer', '0', 'attention', 'self', 'query', 'kernel'), ('encoder', 'layer', '0', 'attention', 'self', 'value', 'bias'), ('encoder', 'la

##### Constrain Dataset
- Arbitrarily filter out perfumes with < 3 reviews
- Arbitrarily retain reviews of 40 to 100 characters in length
- `e5-large-instruct` has a 512-token window size

In [190]:
task = 'A fragrance has these notes:'
notes_to_reviews = dict()

for i, (title, notes, reviews) in enumerate(zip(titles_df['titles'], notes_df['notes'], reviews_df['reviews'])):
    reviews = list(filter(lambda r: len(r) > 40 and len(r) < 100, reviews))
        
    if len(ast.literal_eval(notes)) != 5:
        continue

    if len(reviews) < 3: 
        continue 

    query = [f'{task} {notes}']

    if notes_to_reviews.get(notes): 
        notes_to_reviews[notes] += reviews
    else: 
        notes_to_reviews[notes] = query + reviews

print("Unique Note-Sets: ", len(notes_to_reviews.keys()))
print("N Reviews: ", sum([len(v) for v in notes_to_reviews.values()]) -  len(notes_to_reviews.keys()) )

Unique Note-Sets:  617
N Reviews:  4257


- embedding analysis helper functions

In [ ]:
"""These functions are from the e5-large-instruct HF repo example"""
@jax.jit
def average_pool(last_hidden_states, attention_mask):
    # Expand attention_mask to have the same dimensionality as last_hidden_states
    mask_expanded = attention_mask[..., None]
    # Create a mask of the same shape as last_hidden_states where masked positions are True
    masked_indices = ~mask_expanded.astype(jnp.bool_)
    # Use jnp.where to set masked positions to 0.0
    masked_last_hidden = jnp.where(masked_indices, 0.0, last_hidden_states)
    # Sum over the sequence length dimension (dim=1)
    sum_hidden = jnp.sum(masked_last_hidden, axis=1)
    # Sum the attention mask to get the number of non-padded tokens
    sum_mask = jnp.sum(attention_mask, axis=1)[..., None]
    # Perform the average, handling potential division by zero
    embeddings = sum_hidden / jnp.where(sum_mask == 0, 1, sum_mask)
    return embeddings

@jax.jit
def normalize_vec(x, p=2, axis=-1, epsilon=1e-8):
  norm = jnp.linalg.norm(x, ord=p, axis=axis, keepdims=True)
  return x / (norm + epsilon)

@jax.jit
def similarity_scores(embeddings):
  first_embedding = embeddings[:1]
  remaining_embeddings_transposed = jnp.transpose(embeddings[1:])
  scores = (first_embedding @ remaining_embeddings_transposed) * 100
  return scores

- Toggle `N_SAMPLES` depending on your resources (I was running this on my laptop)
- It takes ~1 sec to generate each embedding on my machine
- Note the distinction between notes-sets and fragrances; multiple fragrances can share the same set of notes.

In [198]:
notes_to_embeddings = {}
notes_to_scores = {}
threads = []
# These samples refer to Note-Sets
MAX_SAMPLES = 24477
N_SAMPLES = len(notes_to_reviews.keys())

for i, key in enumerate(notes_to_reviews.keys()):
  if i == N_SAMPLES: break
  batch_dict = tokenizer(notes_to_reviews[key], max_length=512, padding=True, truncation=True, return_tensors='np')
  outputs = model(**batch_dict)
  embeddings = normalize_vec(average_pool(outputs.last_hidden_state, batch_dict['attention_mask']))
  scores = similarity_scores(embeddings)
  notes_to_embeddings[key] = embeddings[1:] # skips the system prompt
  notes_to_scores[key] = scores[0]

# Save the results
with open('./checkpoints/pangrams2/notes_to_scores.pkl', 'wb') as f1:
  pickle.dump(notes_to_scores, f1)
with open('./checkpoints/pangrams2/notes_to_reviews.pkl', 'wb') as f2:
  pickle.dump(notes_to_reviews, f2)
with open('./checkpoints/pangrams2/notes_to_embeddings.pkl', 'wb') as f3:
  pickle.dump(notes_to_embeddings, f3)

#### Load Review Embeddings (checkpoint)

In [3]:
import pickle 

with open('./checkpoints/pangrams/notes_to_scores.pkl', 'rb') as f1:
  notes_to_scores = pickle.load(f1)
with open('./checkpoints/pangrams/notes_to_reviews.pkl', 'rb') as f2:
  notes_to_reviews = pickle.load(f2)
with open('./checkpoints/pangrams/notes_to_embeddings.pkl', 'rb') as f3:
  notes_to_embeddings = pickle.load(f3)

- randomly select a fragrance and rank reviews by semantic similarity to its notes

In [28]:
import random

key = random.choice(list(notes_to_scores.keys()))
sorted_reviews = sorted(zip(notes_to_scores[key], notes_to_reviews[key][1:]), key=lambda x:x[0], reverse=True)

print("Notes: ", key)
print("Rank\t| Score\t\t| Comment \n-------------------------------------------------------------")
for i, review in enumerate(sorted_reviews):
  print(i + 1, "\t| ", review[0], "\t| ", review[1].replace("\n", "")[:40])

Notes:  ['Cardamom', 'Honey', 'Pepper', 'Siam Benzoin', 'Thanaka Wood']
Rank	| Score		| Comment 
-------------------------------------------------------------
1 	|  90.45498 	|  Spicy-woody-honey gourmand fragrance. Ho
2 	|  88.87873 	|  wow, love at first sniff, resin, woods, 
3 	|  88.169525 	|  This is so Beautiful.First whiff is Card
4 	|  88.03259 	|  Mysterious, unique and beautiful. Meltin
5 	|  86.615005 	|  A lighter version of Bond's New Haarlem.
6 	|  86.59332 	|  Honey, honey, and more honey!! I do not 
7 	|  85.244446 	|  Slightly sweet powder honey with an unde
8 	|  82.6171 	|  Started off beautifully resiney-spicey, 
9 	|  82.251625 	|  God this is so rich, luscious, captivati
10 	|  81.65783 	|  Lovely start, all the right tunes, the h
11 	|  81.44917 	|  One of the most concrete benzoin solifor


- Count unique embeddings

In [19]:
embeddings = set()
for v in notes_to_embeddings.values():
  embeddings.update(set([str(r) for r in v]))

print("Note-Set sample size:\t", len(notes_to_embeddings.keys()))
print("Number of Reviews:\t", sum([len(e) for e in notes_to_embeddings.values()]))
all_unique = sum([len(v) for v in notes_to_embeddings.values()]) ==  len(embeddings)
print("All Reviews Unique?\t", all_unique)

# print repeats
if not all_unique:
  repeats = dict()
  for key, reviews in notes_to_reviews.items():
    for r in reviews:
      if repeats.get(r):
        repeats[r] += 1
      else:
        repeats[r] = 1
  for k, v in repeats.items():
    if v > 1:
      print("Repeated Review:\t", k,"\nNum Repeats:\t", v)

Note-Set sample size:	 1584
Number of Reviews:	 23576
All Reviews Unique?	 False
Repeated Review:	 This brand so fantastic novel direction, you no longer worry about projection, performance. There have options for concentration. 
Num Repeats:	 2
Repeated Review:	 This site promotes hate against LGBTQ people. 
Num Repeats:	 3
Repeated Review:	 Anyone based in the UK able to decant this for me? 
Num Repeats:	 2
Repeated Review:	 This one should be promoted to the main line. 
Retro, but ahead of its time. 
Num Repeats:	 2
Repeated Review:	 Please read this: I value my perfume collection very much, but some of the perfumes are missing the cap - so if you have emptied your perfume, I will be very happy and grateful if I can ask for your cap (size 75 ml)
Please send me a PM - Thank you. 
Num Repeats:	 2
Repeated Review:	 WARNING!!! if you don't want to loose your money- don't buy from them! This company is a fraud, they take money but never ship!! 
Num Repeats:	 2
Repeated Review:	 Just a he

##### MLP

In [21]:
import jax
from typing import Any, Callable, Sequence
from jax import random, numpy as jnp
import flax
from flax import linen as nn
import sys

class MLP(nn.Module):
  features: Sequence[int]
  @nn.compact
  def __call__(self, inputs):
    x = inputs
    for i, feat in enumerate(self.features):
      x = nn.relu(nn.Dense(feat, name=f'layers_{i}')(x))
      if i != len(self.features) - 1:
        x = nn.softmax(x)
    return x

def train_step(params, x, y, learning_rate):
  def loss_fn(params, x, y):
    y_pred = model.apply(params, x)
    loss = jnp.mean((y_pred - y)**2)
    return loss, y_pred

  grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
  (loss, y_pred), grads = grad_fn(params, x, y)
  new_params = jax.tree_util.tree_map(lambda p, g: p - learning_rate * g, params, grads)
  return new_params, loss, y_pred

In [22]:
# --- DATA ---
# {id -> (review embedding, review text, one_hot_enc label, actual label)}
ALL_DATA = dict() 
N_LABELS = len(notes_to_embeddings.keys())
one_hot_enc = jax.nn.one_hot(jnp.array([i for i in range(N_LABELS)]), N_LABELS)
label_to_idx = {label: idx for (idx, label) in enumerate(notes_to_embeddings.keys())}

example_i = 0

for label in notes_to_embeddings.keys():
  # notes_to_reviews is 1-indexed to skip the embedding instruction
  for enc, review in zip(notes_to_embeddings[label], notes_to_reviews[label][1:]):
    ALL_DATA[example_i] = (enc, review, one_hot_enc[label_to_idx[label]], label)
    example_i += 1

shuffled_keys = list(ALL_DATA.keys())
rand.shuffle(shuffled_keys)


TRAIN_DATA = {
  key: ALL_DATA[key] for (i, key) in enumerate(shuffled_keys) if len(
    notes_to_reviews[ALL_DATA[key][3]]
  ) < 10
}

TEST_DATA = {
  key: ALL_DATA[key] for (i, key) in enumerate(shuffled_keys) if i not in TRAIN_DATA and len(
    notes_to_reviews[ALL_DATA[key][3]]
  ) < 10
}

BATCH_SIZE = len(TRAIN_DATA.keys())

# --- HYPERPARAMETERS --- 
N_EPOCHS = 101
LEARNING_RATE = 0.03
REVIEW_EMBD_LEN = list(notes_to_embeddings.values())[0].shape[1]

# --- INITIALIZE MLP ---
key1, key2 = random.split(random.key(0))
x = random.uniform(key1, (BATCH_SIZE, REVIEW_EMBD_LEN,))
model = MLP(features=[REVIEW_EMBD_LEN, 512, 128, N_LABELS])
params = model.init(key2, x[0])
y = jnp.array([example[2] for example in TRAIN_DATA.values()])
print("MLP Shape:", x.shape, " -> ", y.shape)

# --- TRAIN ---
for epoch in range(N_EPOCHS):
  params, loss, y_pred = train_step(params, x, y, LEARNING_RATE)
  print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

# --- SAVE MODEL PARAMS ---
with open('./checkpoints/MLP_Params.pkl', 'wb') as file:
  pickle.dump(params, file)

MLP Shape: (4352, 1024)  ->  (4352, 1584)
Epoch 1, Loss: 0.0007
Epoch 2, Loss: 0.0007
Epoch 3, Loss: 0.0007
Epoch 4, Loss: 0.0007
Epoch 5, Loss: 0.0007
Epoch 6, Loss: 0.0007
Epoch 7, Loss: 0.0007
Epoch 8, Loss: 0.0007
Epoch 9, Loss: 0.0007
Epoch 10, Loss: 0.0007
Epoch 11, Loss: 0.0007
Epoch 12, Loss: 0.0007
Epoch 13, Loss: 0.0007
Epoch 14, Loss: 0.0007
Epoch 15, Loss: 0.0007
Epoch 16, Loss: 0.0007
Epoch 17, Loss: 0.0007
Epoch 18, Loss: 0.0007
Epoch 19, Loss: 0.0007
Epoch 20, Loss: 0.0007
Epoch 21, Loss: 0.0007
Epoch 22, Loss: 0.0007
Epoch 23, Loss: 0.0007
Epoch 24, Loss: 0.0007
Epoch 25, Loss: 0.0007
Epoch 26, Loss: 0.0007
Epoch 27, Loss: 0.0007
Epoch 28, Loss: 0.0007
Epoch 29, Loss: 0.0007
Epoch 30, Loss: 0.0007
Epoch 31, Loss: 0.0007
Epoch 32, Loss: 0.0007
Epoch 33, Loss: 0.0007
Epoch 34, Loss: 0.0007
Epoch 35, Loss: 0.0007
Epoch 36, Loss: 0.0007
Epoch 37, Loss: 0.0007
Epoch 38, Loss: 0.0007
Epoch 39, Loss: 0.0007
Epoch 40, Loss: 0.0007
Epoch 41, Loss: 0.0007
Epoch 42, Loss: 0.0007
E

##### EVALUATE MLP (checkpoint)
- ~10 inferences/sec on my laptop
- For the 5-note-filtered data set, `['Ambrette (Musk Mallow)', 'Cedar', 'Iris', 'Lavender', 'Vetiver']` is predicted for every review (rip)

In [ ]:
# LOAD MODEL PARAMS
with open('./checkpoints/MLP_Params.pkl', 'rb') as file:
  params = pickle.load(file)

idx_to_label = {idx: label for (idx, label) in enumerate(notes_to_embeddings.keys())}
correct = 0

# evaluation run
for trial_i, key in enumerate(TEST_DATA):
  (input, review, _, actual_label) = TEST_DATA[key]
  y = model.apply(params, input)
  prediction = idx_to_label[max(list(zip(y, [i for i in range(len(y))])))[1]]
  
  if actual_label == prediction: 
    correct += 1
  print("-------------")
  print(f"Trial {trial_i}:")
  print("Review: ", review.replace("\n", ""))
  print("Actual: ", actual_label)
  print("Predicted: ", prediction)
  print("Correct? ", actual_label == prediction)

print("Total Correct:\t", correct, "/", len(TEST_DATA.keys()))